# Tutorial 13: Analytical Gradients for Mechanism Optimization

**Prerequisites:** Tutorial 4 — Numerical Gradients, Tutorial 5 — Newton & Quasi-Newton Methods  
**Up Next:** Tutorial 14 — Hybrid Strategies

---

## Concept

In Tutorial 4 we approximated gradients via finite differences. That approach is general-purpose but has two drawbacks:

1. **Accuracy:** Forward differences are $O(\delta)$ and central differences are $O(\delta^2)$. Neither is exact.
2. **Cost:** Each gradient evaluation requires $O(n)$ extra function calls.

For our four-bar linkage, the FK equations are smooth and differentiable — we can derive the gradient *analytically* using the **implicit function theorem**. The result is exact (to machine precision) and costs just one forward pass through a chain of derivatives.

## Key Takeaway

> **Analytical gradients are exact and cheap. When your kinematic equations are differentiable, the implicit function theorem gives you the sensitivity of passive angles to design variables — then the chain rule does the rest. Worth deriving for any production optimization.**

## Math Core

The four-bar FK solves for passive angles $(\theta_2, \theta_3)$ as implicit functions of the design variables $(\ell_2, \ell_3)$. The key insight is the **implicit function theorem**:

If $F(\boldsymbol{\theta}, \mathbf{x}) = 0$ defines $\boldsymbol{\theta}$ implicitly as a function of $\mathbf{x}$, then:

$$\frac{d\boldsymbol{\theta}}{d\mathbf{x}} = -\left(\frac{\partial F}{\partial \boldsymbol{\theta}}\right)^{-1} \frac{\partial F}{\partial \mathbf{x}}$$

For the coupler point $\mathbf{p}(\boldsymbol{\theta}, \mathbf{x})$, the total derivative is:

$$\frac{d\mathbf{p}}{d\mathbf{x}} = \frac{\partial \mathbf{p}}{\partial \mathbf{x}} + \frac{\partial \mathbf{p}}{\partial \boldsymbol{\theta}} \frac{d\boldsymbol{\theta}}{d\mathbf{x}}$$

We will work through each piece step by step.

## Code

### Setup: the four-bar problem (same as all tutorials)

In [ ]:
from dms.mechanisms.fourbar import FourBar
from dms.curves import CompareCurves
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
%matplotlib inline

# Fixed link lengths
L0, L1 = 1.0, 1.0
# Coupler point offset
PX, PY = 0.5, 0.3
# Target path
TARGETS = np.array([
    [1.0, 1.5], [0.75, 1.5], [0.5, 1.5],
    [0.25, 1.5], [0.0, 1.5], [-0.25, 1.5]
])
# Crank angles
THETAS = np.linspace(np.deg2rad(60), np.deg2rad(120), len(TARGETS))

In [ ]:
def fourbar_fk(theta1, l0, l1, l2, l3):
    """Cosine-law closed-form FK for four-bar linkage."""
    ax, ay = l1 * np.cos(theta1), l1 * np.sin(theta1)
    dx, dy = ax - l0, ay
    d = np.sqrt(dx**2 + dy**2)
    cos_alpha = (l3**2 + d**2 - l2**2) / (2 * l3 * d)
    if np.abs(cos_alpha) > 1:
        return None
    alpha = np.arccos(np.clip(cos_alpha, -1, 1))
    beta = np.arctan2(dy, dx)
    theta3 = beta + alpha
    bx = l0 + l3 * np.cos(theta3)
    by = l3 * np.sin(theta3)
    theta2 = np.arctan2(by - ay, bx - ax)
    return theta2, theta3


def coupler_point(theta1, l2, l3):
    """Compute coupler point for given crank angle."""
    sol = fourbar_fk(theta1, L0, L1, l2, l3)
    if sol is None:
        return None
    theta2, _ = sol
    ax, ay = L1 * np.cos(theta1), L1 * np.sin(theta1)
    ux, uy = np.cos(theta2), np.sin(theta2)
    return np.array([ax + PX * ux - PY * uy,
                     ay + PX * uy + PY * ux])


def objective(x):
    """Path-synthesis objective."""
    l2, l3 = x
    path = []
    for theta1 in THETAS:
        pt = coupler_point(theta1, l2, l3)
        if pt is None:
            return 1e3
        path.append(pt)
    path = np.array(path)
    return CompareCurves(path[:, 0], path[:, 1],
                         TARGETS[:, 0], TARGETS[:, 1])

### Step 1: The FK chain and its intermediate quantities

Our FK computes $\theta_3$ and $\theta_2$ from $(\ell_2, \ell_3)$ through a chain of intermediate values. Let's lay out the computation graph:

1. $A = (\ell_1 \cos\theta_1,\; \ell_1 \sin\theta_1)$ — crank tip (independent of $\ell_2, \ell_3$)
2. $d_x = a_x - \ell_0, \quad d_y = a_y$ — vector from D to A
3. $d = \sqrt{d_x^2 + d_y^2}$ — distance D to A (independent of $\ell_2, \ell_3$)
4. $\cos\alpha = \frac{\ell_3^2 + d^2 - \ell_2^2}{2 \ell_3 d}$ — cosine law
5. $\alpha = \arccos(\cos\alpha)$
6. $\beta = \text{atan2}(d_y, d_x)$ — (independent of $\ell_2, \ell_3$)
7. $\theta_3 = \beta + \alpha$
8. $B = (\ell_0 + \ell_3 \cos\theta_3,\; \ell_3 \sin\theta_3)$
9. $\theta_2 = \text{atan2}(b_y - a_y,\; b_x - a_x)$
10. Coupler point: $\mathbf{p} = A + P_x \hat{u} + P_y \hat{v}$ where $\hat{u} = (\cos\theta_2, \sin\theta_2)$, $\hat{v} = (-\sin\theta_2, \cos\theta_2)$

We need $\frac{d\mathbf{p}}{d\ell_2}$ and $\frac{d\mathbf{p}}{d\ell_3}$. Since $A$, $d$, $\beta$ don't depend on the design variables, the chain starts at step 4.

### Step 2: Derivatives of $\cos\alpha$ w.r.t. $\ell_2$ and $\ell_3$

$$\cos\alpha = \frac{\ell_3^2 + d^2 - \ell_2^2}{2 \ell_3 d}$$

$$\frac{\partial \cos\alpha}{\partial \ell_2} = \frac{-2\ell_2}{2 \ell_3 d} = \frac{-\ell_2}{\ell_3 d}$$

For $\ell_3$, use the quotient rule. Let $N = \ell_3^2 + d^2 - \ell_2^2$ and $D = 2\ell_3 d$:

$$\frac{\partial \cos\alpha}{\partial \ell_3} = \frac{2\ell_3 \cdot 2\ell_3 d - N \cdot 2d}{(2\ell_3 d)^2} = \frac{2\ell_3}{2\ell_3 d} - \frac{N}{2\ell_3^2 d} = \frac{1}{d} - \frac{\ell_3^2 + d^2 - \ell_2^2}{2\ell_3^2 d}$$

Simplifying:

$$\frac{\partial \cos\alpha}{\partial \ell_3} = \frac{2\ell_3^2 - (\ell_3^2 + d^2 - \ell_2^2)}{2\ell_3^2 d} = \frac{\ell_3^2 - d^2 + \ell_2^2}{2\ell_3^2 d}$$

### Step 3: Derivative of $\alpha$ and $\theta_3$

Since $\alpha = \arccos(\cos\alpha)$:

$$\frac{d\alpha}{d(\cos\alpha)} = \frac{-1}{\sin\alpha}$$

And since $\theta_3 = \beta + \alpha$ and $\beta$ is independent of $\ell_2, \ell_3$:

$$\frac{\partial \theta_3}{\partial \ell_k} = \frac{\partial \alpha}{\partial \ell_k} = \frac{-1}{\sin\alpha} \frac{\partial \cos\alpha}{\partial \ell_k}$$

### Step 4: Derivative of $\theta_2$

We have $\theta_2 = \text{atan2}(b_y - a_y,\; b_x - a_x)$ where:
- $b_x = \ell_0 + \ell_3 \cos\theta_3$, $\quad b_y = \ell_3 \sin\theta_3$
- $a_x, a_y$ are independent of $\ell_2, \ell_3$

Let $r_x = b_x - a_x$ and $r_y = b_y - a_y$, so $r^2 = r_x^2 + r_y^2$ (note: $r = \ell_2$, the coupler length!).

For $\text{atan2}(r_y, r_x)$:

$$\frac{d\theta_2}{dr_x} = \frac{-r_y}{r^2}, \quad \frac{d\theta_2}{dr_y} = \frac{r_x}{r^2}$$

We need $\frac{\partial r_x}{\partial \ell_k}$ and $\frac{\partial r_y}{\partial \ell_k}$:

$$\frac{\partial b_x}{\partial \ell_2} = -\ell_3 \sin\theta_3 \cdot \frac{\partial \theta_3}{\partial \ell_2}, \quad \frac{\partial b_y}{\partial \ell_2} = \ell_3 \cos\theta_3 \cdot \frac{\partial \theta_3}{\partial \ell_2}$$

$$\frac{\partial b_x}{\partial \ell_3} = \cos\theta_3 - \ell_3 \sin\theta_3 \cdot \frac{\partial \theta_3}{\partial \ell_3}, \quad \frac{\partial b_y}{\partial \ell_3} = \sin\theta_3 + \ell_3 \cos\theta_3 \cdot \frac{\partial \theta_3}{\partial \ell_3}$$

Then:

$$\frac{\partial \theta_2}{\partial \ell_k} = \frac{1}{r^2}\left(-r_y \frac{\partial r_x}{\partial \ell_k} + r_x \frac{\partial r_y}{\partial \ell_k}\right)$$

### Step 5: Derivative of the coupler point

The coupler point is:

$$\mathbf{p} = \begin{pmatrix} a_x + P_x \cos\theta_2 - P_y \sin\theta_2 \\ a_y + P_x \sin\theta_2 + P_y \cos\theta_2 \end{pmatrix}$$

Since $a_x, a_y$ are independent of $\ell_2, \ell_3$:

$$\frac{\partial \mathbf{p}}{\partial \ell_k} = \frac{\partial \theta_2}{\partial \ell_k} \begin{pmatrix} -P_x \sin\theta_2 - P_y \cos\theta_2 \\ P_x \cos\theta_2 - P_y \sin\theta_2 \end{pmatrix}$$

This is the full analytical derivative of the coupler point position w.r.t. each design variable, computed through the chain: $\ell_k \to \cos\alpha \to \alpha \to \theta_3 \to B \to \theta_2 \to \mathbf{p}$.

### Implementation: `analytical_gradient`

Now we implement the full chain in numpy.

In [ ]:
def coupler_point_and_grad(theta1, l2, l3):
    """Compute coupler point AND its gradient w.r.t. [l2, l3].
    
    Returns:
        point: (2,) array [px, py]
        grad:  (2, 2) array where grad[i, j] = d(point_i) / d(x_j)
               columns are [d/dl2, d/dl3]
    Returns (None, None) if mechanism can't assemble.
    """
    # --- Forward pass (same as fourbar_fk + coupler_point) ---
    ax, ay = L1 * np.cos(theta1), L1 * np.sin(theta1)
    ddx, ddy = ax - L0, ay
    d = np.sqrt(ddx**2 + ddy**2)
    
    cos_alpha = (l3**2 + d**2 - l2**2) / (2 * l3 * d)
    if np.abs(cos_alpha) > 1:
        return None, None
    
    alpha = np.arccos(np.clip(cos_alpha, -1, 1))
    sin_alpha = np.sin(alpha)
    if np.abs(sin_alpha) < 1e-15:
        return None, None  # degenerate configuration
    
    beta = np.arctan2(ddy, ddx)
    theta3 = beta + alpha
    
    bx = L0 + l3 * np.cos(theta3)
    by = l3 * np.sin(theta3)
    rx, ry = bx - ax, by - ay
    r2 = rx**2 + ry**2  # = l2^2
    
    theta2 = np.arctan2(ry, rx)
    
    # Coupler point
    c2, s2 = np.cos(theta2), np.sin(theta2)
    px = ax + PX * c2 - PY * s2
    py = ay + PX * s2 + PY * c2
    point = np.array([px, py])
    
    # --- Backward pass: analytical derivatives ---
    
    # d(cos_alpha)/dl2, d(cos_alpha)/dl3
    dca_dl2 = -l2 / (l3 * d)
    dca_dl3 = (l3**2 - d**2 + l2**2) / (2 * l3**2 * d)
    
    # d(alpha)/dl = d(alpha)/d(cos_alpha) * d(cos_alpha)/dl
    dalpha_dca = -1.0 / sin_alpha
    dalpha_dl2 = dalpha_dca * dca_dl2
    dalpha_dl3 = dalpha_dca * dca_dl3
    
    # d(theta3)/dl = d(alpha)/dl  (beta independent of l2, l3)
    dt3_dl2 = dalpha_dl2
    dt3_dl3 = dalpha_dl3
    
    # d(B)/dl
    c3, s3 = np.cos(theta3), np.sin(theta3)
    
    dbx_dl2 = -l3 * s3 * dt3_dl2
    dby_dl2 =  l3 * c3 * dt3_dl2
    
    dbx_dl3 = c3 + (-l3 * s3) * dt3_dl3
    dby_dl3 = s3 + ( l3 * c3) * dt3_dl3
    
    # d(r)/dl = d(B)/dl  (A is independent of l2, l3)
    drx_dl2 = dbx_dl2
    dry_dl2 = dby_dl2
    drx_dl3 = dbx_dl3
    dry_dl3 = dby_dl3
    
    # d(theta2)/dl = (1/r^2) * (-ry * drx/dl + rx * dry/dl)
    dt2_dl2 = (-ry * drx_dl2 + rx * dry_dl2) / r2
    dt2_dl3 = (-ry * drx_dl3 + rx * dry_dl3) / r2
    
    # d(coupler_point)/dl = d(theta2)/dl * [-PX*sin(t2) - PY*cos(t2),
    #                                        PX*cos(t2) - PY*sin(t2)]
    dp_dt2 = np.array([-PX * s2 - PY * c2,
                        PX * c2 - PY * s2])
    
    # grad[:, 0] = dp/dl2, grad[:, 1] = dp/dl3
    grad = np.column_stack([dp_dt2 * dt2_dl2,
                            dp_dt2 * dt2_dl3])
    
    return point, grad

### Validate against finite differences

The acid test: do the analytical derivatives match numerical ones to near machine precision?

In [ ]:
# Test at a specific configuration
l2_test, l3_test = 1.5, 1.5
theta1_test = np.deg2rad(90)

pt, grad_ana = coupler_point_and_grad(theta1_test, l2_test, l3_test)

# Finite difference gradient for comparison
eps = 1e-7
pt_l2p = coupler_point(theta1_test, l2_test + eps, l3_test)
pt_l2m = coupler_point(theta1_test, l2_test - eps, l3_test)
pt_l3p = coupler_point(theta1_test, l2_test, l3_test + eps)
pt_l3m = coupler_point(theta1_test, l2_test, l3_test - eps)

grad_fd = np.column_stack([
    (pt_l2p - pt_l2m) / (2 * eps),
    (pt_l3p - pt_l3m) / (2 * eps)
])

print('Analytical gradient:')
print(grad_ana)
print('\nFinite-difference gradient:')
print(grad_fd)
print(f'\nMax absolute difference: {np.max(np.abs(grad_ana - grad_fd)):.2e}')

In [ ]:
# Validate across multiple configurations
print(f'{"theta1 (deg)":>12}  {"max |ana - fd|":>16}  {"status"}')
print('-' * 50)
for t1 in THETAS:
    pt_a, g_a = coupler_point_and_grad(t1, l2_test, l3_test)
    if pt_a is None:
        continue
    g_fd = np.column_stack([
        (coupler_point(t1, l2_test + eps, l3_test) -
         coupler_point(t1, l2_test - eps, l3_test)) / (2 * eps),
        (coupler_point(t1, l2_test, l3_test + eps) -
         coupler_point(t1, l2_test, l3_test - eps)) / (2 * eps)
    ])
    err = np.max(np.abs(g_a - g_fd))
    status = 'OK' if err < 1e-6 else 'MISMATCH'
    print(f'{np.rad2deg(t1):12.1f}  {err:16.2e}  {status}')

The analytical and finite-difference gradients agree to roughly $10^{-8}$ or better — the remaining difference is the $O(\delta^2)$ truncation error from central differences, not an error in our derivation.

### Gradient of the objective function

The objective sums `CompareCurves` over all crank angles. `CompareCurves` returns the sum of squared distances:

$$f = \sum_i \| \mathbf{p}_i - \mathbf{t}_i \|^2$$

So:

$$\frac{\partial f}{\partial \mathbf{x}} = \sum_i 2 (\mathbf{p}_i - \mathbf{t}_i)^T \frac{\partial \mathbf{p}_i}{\partial \mathbf{x}}$$

In [ ]:
def analytical_gradient(x):
    """Analytical gradient of the path-synthesis objective w.r.t. [l2, l3]."""
    l2, l3 = x
    grad = np.zeros(2)
    for i, theta1 in enumerate(THETAS):
        pt, dp = coupler_point_and_grad(theta1, l2, l3)
        if pt is None:
            return np.array([0.0, 0.0])  # infeasible — no gradient info
        residual = pt - TARGETS[i]  # (2,)
        # d/dx of ||pt - target||^2 = 2 * residual^T @ dp/dx
        grad += 2.0 * residual @ dp  # (2,)
    return grad

In [ ]:
# Validate objective gradient against finite differences
x_test = np.array([1.5, 1.5])
g_analytical = analytical_gradient(x_test)

eps = 1e-7
g_fd = np.array([
    (objective(x_test + [eps, 0]) - objective(x_test - [eps, 0])) / (2 * eps),
    (objective(x_test + [0, eps]) - objective(x_test - [0, eps])) / (2 * eps)
])

print(f'Analytical gradient: [{g_analytical[0]:+.8f}, {g_analytical[1]:+.8f}]')
print(f'FD gradient:         [{g_fd[0]:+.8f}, {g_fd[1]:+.8f}]')
print(f'Max difference:      {np.max(np.abs(g_analytical - g_fd)):.2e}')

### BFGS with analytical gradients vs. finite differences

Now the payoff: let's compare L-BFGS-B using our analytical gradient against the default finite-difference approximation.

In [ ]:
import time

x0 = np.array([1.0, 1.5])
bounds = [(0.3, 2.5), (0.3, 2.5)]

# --- BFGS with finite differences (default) ---
fd_path = [x0.copy()]
fd_nfev = [0]

def obj_counting_fd(x):
    fd_nfev[0] += 1
    return objective(x)

t0 = time.perf_counter()
res_fd = minimize(obj_counting_fd, x0, method='L-BFGS-B',
                  bounds=bounds,
                  callback=lambda xk: fd_path.append(xk.copy()),
                  options={'ftol': 1e-12, 'eps': 1e-7})
time_fd = time.perf_counter() - t0
fd_path = np.array(fd_path)

# --- BFGS with analytical gradient ---
ana_path = [x0.copy()]
ana_nfev = [0]

def obj_counting_ana(x):
    ana_nfev[0] += 1
    return objective(x)

t0 = time.perf_counter()
res_ana = minimize(obj_counting_ana, x0, method='L-BFGS-B',
                   jac=analytical_gradient,
                   bounds=bounds,
                   callback=lambda xk: ana_path.append(xk.copy()),
                   options={'ftol': 1e-12})
time_ana = time.perf_counter() - t0
ana_path = np.array(ana_path)

print('Method                Iterations  f(x*)       f evals   Time (ms)')
print('-' * 72)
print(f'BFGS + FD gradient    {len(fd_path)-1:>5d}       '
      f'{res_fd.fun:.6f}    {fd_nfev[0]:>5d}     {time_fd*1000:.1f}')
print(f'BFGS + analytical     {len(ana_path)-1:>5d}       '
      f'{res_ana.fun:.6f}    {ana_nfev[0]:>5d}     {time_ana*1000:.1f}')

In [ ]:
# Convergence comparison
fd_costs = [objective(p) for p in fd_path]
ana_costs = [objective(p) for p in ana_path]

fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(range(len(fd_costs)), fd_costs, 'b-o', ms=4, lw=2,
            label=f'BFGS + FD ({len(fd_path)-1} iters)')
ax.semilogy(range(len(ana_costs)), ana_costs, 'r-s', ms=5, lw=2,
            label=f'BFGS + analytical ({len(ana_path)-1} iters)')
ax.set_xlabel('Iteration')
ax.set_ylabel(r'$f(\mathbf{x})$')
ax.set_title('BFGS convergence: analytical vs. finite-difference gradients')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()

### Trajectories on the contour plot

In [ ]:
l2v = np.linspace(0.3, 2.5, 40)
l3v = np.linspace(0.3, 2.5, 40)
L2g, L3g = np.meshgrid(l2v, l3v)
Z = np.vectorize(lambda a, b: objective([a, b]))(L2g, L3g)

fig, ax = plt.subplots(figsize=(7, 5))
cf = ax.contourf(L2g, L3g, np.log1p(Z), levels=30, cmap='viridis')
ax.plot(fd_path[:, 0], fd_path[:, 1], 'b.-', ms=5, lw=1.5,
        label='BFGS + FD')
ax.plot(ana_path[:, 0], ana_path[:, 1], 'r.-', ms=6, lw=2,
        label='BFGS + analytical')
ax.plot(*x0, 'wo', ms=8, zorder=5, label='start')
ax.set_xlabel(r'$\ell_2$')
ax.set_ylabel(r'$\ell_3$')
ax.set_title('Optimization trajectories: analytical vs. FD gradients')
ax.legend()
plt.colorbar(cf, label=r'$\ln(1 + f)$')
plt.tight_layout()

### Visualizing the optimized mechanism

In [ ]:
l2_opt, l3_opt = res_ana.x
mechanism = FourBar(L0, L1, l2_opt, l3_opt)
path = np.array([coupler_point(t, l2_opt, l3_opt) for t in THETAS])

fig, ax = plt.subplots(figsize=(6, 5))
mechanism.plot(np.deg2rad(90), ax=ax)
ax.plot(TARGETS[:, 0], TARGETS[:, 1], 'r*', ms=10, label='targets')
ax.plot(path[:, 0], path[:, 1], 'b.-', label='coupler path')
ax.set_aspect('equal')
ax.legend()
ax.set_title(f'Analytical-gradient BFGS: $\\ell_2$={l2_opt:.3f}, '
             f'$\\ell_3$={l3_opt:.3f}  |  f = {res_ana.fun:.4f}')
plt.tight_layout()

### Preview: automating this with `dms`

Deriving analytical gradients by hand is instructive but tedious, especially for more complex mechanisms. A planned enhancement to the `dms` library will use **sympy symbolic differentiation** to automate this process:

1. Keep link lengths as symbolic variables in the loop closure equations
2. Differentiate symbolically using the implicit function theorem
3. Lambdify the resulting Jacobians for fast numeric evaluation

The API would look something like:
```python
mechanism = FourBar(L0, L1, l2, l3)
dpos_dl = mechanism.gradient('marker_1', theta1, z, wrt=[2, 3])
```

Until then, the manual approach shown here gives you exact gradients that make BFGS significantly more efficient.

---

## Exercises

1. **Extend to more design variables.** What if $\ell_1$ is also a design variable? Derive $\partial \mathbf{p} / \partial \ell_1$. Hint: now $A$, $d$, and $\beta$ all depend on $\ell_1$.

2. **Second derivatives.** Differentiate `coupler_point_and_grad` one more time to get the Hessian $\nabla^2 f$. Use it in a pure Newton step and compare convergence to BFGS.

3. **Timing study.** For $n$ design variables, analytical gradients cost $O(1)$ extra per variable (just more terms in the chain rule), while finite differences cost $O(n)$ extra function evaluations. Profile the crossover point: at what $n$ does the analytical approach save wall time?